## Multithreading

In [ ]:
'''
Threads 

A thead is a lightweight unit of execution with in a process. In Python, a thread executes
python bytecode and multipe threads can execute concurrently. 

Threads running in a process shares the same memory.
'''

'''
Can thread run in parallel?
- This depends on what you're talking about.

Let know what is GIL(Global interpreter lock)
- Cpython allow only one thread to execute the python bytecode with in a
process beacuse of GIL for efficient memory management.
- So for CPU bound tasks they run concurrently take turn rather than executing python
bytecode in true parallelism on multiple CPU cores.
'''

In [7]:
import threading
import time

def task(n):
    print(f"Running task {n}")
    time.sleep(10)
    print(f'Done task {n}')

thread1 = threading.Thread(target=task, args=(1,))
thread2 = threading.Thread(target=task, args=(2,))
thread3 = threading.Thread(target=task, args=(3,))

thread1.start()
thread2.start()
thread3.start()

thread1.join()
thread2.join()
thread3.join() 

print('Main thread done')

# join() blocks the calling thread until the target thread completes.


Running task 1
Running task 2
Running task 3
Done task 1
Done task 2
Done task 3
Main thread done


In [9]:
## Life cycle of the threads

'''

NEW → RUNNABLE/READY → RUNNING → WAITING/BLOCKED → RUNNING → TERMINATED

Thread()  → creates thread object
start()   → starts the thread
run()     → contains/executes the target work
join()    → waits for thread completion
'''

'\n\nNEW → RUNNABLE/READY → RUNNING → WAITING/BLOCKED → RUNNING → TERMINATED\n\nThread()  → creates thread object\nstart()   → starts the thread\nrun()     → contains/executes the target work\njoin()    → waits for thread completion\n'

In [10]:
'''
Main thread vs worker thread

                    Process
                       │
                 Main Thread
                  /    |    \
                 /     |     \
                ▼      ▼      ▼
            Worker1 Worker2 Worker3
                │      │      │
              task() task() task()

The main thread:

- Creates workers
- Starts workers
- Potentially continues doing other work
- Calls join() if it needs to wait for them
- Eventually finishes

'''

'''
Start vs run

- Start -> Creates threads runs the task in the thread by calling run() internally
- run -> it runs the task in the current thread.
'''

'\nStart vs run\n\n- Start -> Creates threads runs the task in the thread by calling run() internally\n- run -> it runs the task in the current thread.\n'

In [ ]:
'''
Daemon threads;

- A daemon thread is a background thread that run along side the non-daemon threads(Includes main thread).
once the all the non daemon threads are therminated the the daemon thread also get terminated regaedless
of whether its task is complete.

Mostly used in the background clean ups, Monitoring.

thread = threading.Thead(task, args, daemon=True)

Note: mian thread is also non-daemon thread, son the entire code execution is done
daemon thread is also terminated regaedless of whether its task is complete

0 sec → Non daemon thread 1 + Daemon running
1 sec → Non daemon thread 1 + Daemon running
0 sec → Non daemon thread 2 + Daemon running
1 sec → Non daemon thread 2 + Daemon running
3 sec → Main + Daemon running
4 sec → Main finishes
         ↓
      Process exits
         ↓
      Daemon stops
'''



'\nDemon threads;\n\n- A demon thread is a background thread that run along side the non-demon threads(Includes main thread).\nonce the all the non demon threads are therminated the the demon thread also get terminated regaedless\nof whether its task is complete.\n\nMostly used in the background clean ups, Monitoring.\n\nthread = threading.Thead(task, args, demon=True)\n\nNote: mian thread is also non-demon thread, son the entire code execution is done\ndemon thread is also terminated regaedless of whether its task is complete\n'

In [27]:
'''
Thread synchronization primitives
- The main idea is that they help multiple threads coordinate access 
to shared resources or coordinate their execution.

Learn:

- Lock - Only one thread should access a critical section at a time.
- RLock
- Semaphore
- Event
- Condition
- Barrier

'''

# Lock

# - Lock provide mutula exclusion - only thread locks and modifys the variable/resource

# - Bank balance

import threading
import time

balance  = 10000
lock = threading.Lock()

def withdraw(amount):
    global balance
    time.sleep(5)
    print('Withdrawing')
    with lock:
        if balance >= amount:
            balance -= amount
    
    print('Withdrawing done')

def add_pf(amount):
    global balance
    time.sleep(5)
    print('adding pf')
    with lock:
        balance += amount
    
    print('adding pf done')

def check_bal():
    for i in range(5):
        time.sleep(2)
        print(f"Balance {balance}")

#d_thread = threading.Thread(target=check_bal, daemon=True)
thread1 = threading.Thread(target=withdraw, args=(2000,)) 
thread2 = threading.Thread(target=add_pf, args=(1200,))

#d_thread.start()
thread1.start()
thread2.start()

thread1.join()
thread2.join()


print(balance) # 9200 - expected

print('Main thread done')


Withdrawing
Withdrawing done
adding pf
adding pf done
9200
Main thread done


In [ ]:
## RLock

#- The same thread can acquire an RLock multiple times without getting blocked by itself.

class Bank:

    def __init__(self):
        self.lock = threading.RLock()

    def withdraw(self):
        with self.lock:
            self.validate()

    def validate(self):
        with self.lock:
            print("Validating")

'''
withdraw()
   ↓
acquire RLock → count 1
   ↓
validate()
   ↓
acquire RLock → count 2
   ↓
validate finishes
   ↓
release → count 1
   ↓
withdraw finishes
   ↓
release → count 0
   ↓
Lock available
'''

In [28]:
# Semaphore

# - A Semaphore is a synchronization mechanism that allows a limited number of
# threads to access a resource at the same time.

import threading
import time

semaphore = threading.Semaphore(3)

def access_resource(thread_id):
    with semaphore:
        print(f"Thread {thread_id} accessing resource")
        time.sleep(3)
        print(f"Thread {thread_id} finished")

threads = []

for i in range(5):
    t = threading.Thread(
        target=access_resource,
        args=(i,)
    )
    threads.append(t)
    t.start()

for t in threads:
    t.join()

Thread 0 accessing resource
Thread 1 accessing resource
Thread 2 accessing resource
Thread 0 finished
Thread 1 finished
Thread 3 accessing resource
Thread 2 finished
Thread 4 accessing resource
Thread 3 finished
Thread 4 finished


In [30]:
## Event

# - An Event is a synchronization primitive used for communication/signaling between threads.
# - One thread sends a signal, and other threads wait for that signal.
# - Flaging


import threading
import time

db_ready = threading.Event() # Eevnt

def worker():
    print("Worker waiting for DB...")
    
    db_ready.wait() # Event -  waits for db initalization
    
    print("DB ready. Starting work...")

def initialize_database():
    print("Initializing database...")
    time.sleep(3)
    
    print("Database initialized")
    db_ready.set() # Event - Good to start working

    # More work AFTER signaling
    time.sleep(5)
    print("Thread 2 completely finished")

t1 = threading.Thread(target=worker)
t2 = threading.Thread(target=initialize_database)

t1.start()
t2.start()

t1.join()
t2.join()

"""
- Thread 1 and Thread 2 are started one after another by the main thread.
- Thread 1 needs the database to be initialized before it can do its work.
- So Thread 1 calls db_ready.wait() and waits for the event signal.
- Thread 2 initializes the database.
- Once the database is initialized, Thread 2 calls db_ready.set().
- Calling set() signals Thread 1 that the database is ready.
- Thread 1 then continues its work.
- Thread 1 does NOT actually wait for Thread 2 to finish.
  It only waits until Thread 2 signals that the database is ready.
- In this example, db_ready.set() happens at the end of Thread 2's work,
  so Thread 1 appears to continue only after Thread 2 finishes.
  """

Worker waiting for DB...
Initializing database...
Database initialized
DB ready. Starting work...
Thread 2 completely finished


"\n- Thread 1 and Thread 2 are started one after another by the main thread.\n- Thread 1 needs the database to be initialized before it can do its work.\n- So Thread 1 calls db_ready.wait() and waits for the event signal.\n- Thread 2 initializes the database.\n- Once the database is initialized, Thread 2 calls db_ready.set().\n- Calling set() signals Thread 1 that the database is ready.\n- Thread 1 then continues its work.\n- Thread 1 does NOT actually wait for Thread 2 to finish.\n  It only waits until Thread 2 signals that the database is ready.\n- In this example, db_ready.set() happens at the end of Thread 2's work,\n  so Thread 1 appears to continue only after Thread 2 finishes.\n  "

In [1]:
## Contditions

"""
A contitions is synchronized primitive used when a thread needs to wait untill some specific condition becomes true, 
and another thread notifies it that the condition may now be satisfied.

Event: “Something happened.”
Condition: “Something happened, now check whether the condition you need is satisfied.”
"""

import threading
import time

condition = threading.Condition()
data_ready = False

def worker():
    global data_ready

    with condition:
        print("Worker: waiting for data...")

        while not data_ready:
            condition.wait()

        print("Worker: data is ready, processing...")

def producer():
    global data_ready

    time.sleep(3)

    with condition:
        data_ready = True
        print("Producer: data is ready")
        condition.notify()

t1 = threading.Thread(target=worker)
t2 = threading.Thread(target=producer)

t1.start()
t2.start()

t1.join()
t2.join()

Worker: waiting for data...
Producer: data is ready
Worker: data is ready, processing...


In [ ]:
"""
submit() and Future

submit() schedules the task and immediately gives you a Future object.

result() gives you the task's return value
"""


from concurrent.futures import ThreadPoolExecutor
import time

def task(name):
    print(f"{name} started")
    time.sleep(2)
    print(f"{name} completed")
    return f"{name} result"

def task2(name):
    print(f"{name} started")
    time.sleep(4)
    print(f"{name} completed")
    return f"{name} result"

with ThreadPoolExecutor(max_workers=3) as executor:
    #Differnet tasks
    future1 = executor.submit(task, "Task 1")
    future2 = executor.submit(task2, "Task 2")
    future3 = executor.submit(task, "Task 3")

    print(future1.result())
    print(future2.result())
    print(future3.result())

Task 1 started
Task 2 started
Task 3 started
Task 1 completed
Task 3 completed
Task 1 result
Task 2 completed
Task 2 result
Task 3 result


In [5]:
from concurrent.futures import ThreadPoolExecutor
import time

def task(name):
    print(f"{name} started")
    time.sleep(2)
    print(f"{name} completed")
    return f"{name} result"

tasks = ['task1', 'task2', 'task3']

with ThreadPoolExecutor(max_workers=3) as executor:

    results = executor.map(task, tasks)

    for result in results:
        print(result)
   

task1 started
task2 started
task3 started
task1 completed
task1 result
task2 completed
task2 result
task3 completed
task3 result
